Mevsim kalibrasyonu (sub_15 ve -sub_16? denenmeyebilir-)

Kısaca ne yapıyoruz

Modelimiz yazın tüketimin ne kadar arttığını yetersiz tahmin ediyor. Biraz bu case'e yöneleceğim.
Bunu train verisiyle ölçüp tahminleri ay bazında kaydırıyoruz.

sub_10'un (en iyi skorun üstüne) tahminlerini alıyoruz, her aya bir
düzeltme katsayısı uyguluyoruz, yeni dosyayı kaydediyoruz. Cold-start'a dokunmuyorum.


Test dönemi Nisan-Temmuz (yaz). Modelin elindeki en taze profil bilgisi
ise son30 / son90 özellikleri, yani Ocak-Mart (kış). Model kış
seviyesini alıp global bir mevsim düzeltmesi uyguluyor. Ama mevsimsellik
trafodan trafoya çok değişiyor: Gördes'teki bir trafo yazın kışın 4.6 katı
tüketirken (tarımsal sulama kaynaklı olabilir bilgisi var), Bornova'daki bir trafo yazın daha az tüketiyor.
Model ikisine de aynı ortalama eğriyi uyguladığı için eğri düzleşiyor.

Uygulanacak adımları:

1. Teşhis — 2025'in gerçek yaz artışıyla bizim 2026 tahminimizi karşılaştır
2. Sebep — trafo bazlı mevsimsellik ne kadar değişken, ölç
3. Düzeltme — aylık offset'leri hesapla
4. Kazanç — düzeltmenin skora etkisini tahmin et
5. Üretim — sub_15 (tam düzeltme) ve sub_16 (kısmi düzeltme) dosyaları

In [7]:
import pandas as pd
import numpy as np

RAW = '../data/raw/'
TABAN = 'sub_10_kurguA_sub02_origin'   # şu anki en iyi skor: 1.05487

train = pd.read_csv(RAW + 'train.csv', parse_dates=['tarih'])
test = pd.read_csv(RAW + 'test.csv', parse_dates=['tarih'])
taban_sub = pd.read_csv(f'../submissions/{TABAN}.csv')

train['ay'] = train['tarih'].dt.month
train['yil'] = train['tarih'].dt.year
test['ay'] = test['tarih'].dt.month

# tahminleri test satırlarıyla birleştir, log uzayında çalışacağız
te = test.merge(taban_sub, on='id')
te['log_tahmin'] = np.log1p(te['tuketim'])

print(f'taban submission : {TABAN} (Kaggle 1.05487)')
print(f'train  : {train.tarih.min().date()} -> {train.tarih.max().date()}')
print(f'test   : {test.tarih.min().date()} -> {test.tarih.max().date()}')
print(f'test ayları ve satır sayıları:')
print(te.groupby('ay').size().to_string())

taban submission : sub_10_kurguA_sub02_origin (Kaggle 1.05487)
train  : 2025-01-01 -> 2026-03-31
test   : 2026-04-01 -> 2026-07-31
test ayları ve satır sayıları:
ay
4    117604
5    182673
6    203279
7    211132


1. Teşhis: yaz artışını yakayabiliyor muyuz / ne oranda?

Doğrudan karşılaştırılamaz çünkü 2026 yazının gerçeği elimizde yok. Ama
şekli karşılaştırabiliriz. Her iki yıl için de Mart ayını referans alıp
"Mart'a göre kaç log yükseldi" sorusunu soruyoruz:

- 2025 için: gerçek veriden (train'de var)
- 2026 için: kendi tahminlerimizden

Mevsimsellik fiziksel bir olay (klima, sulama) olduğu için iki yılın şekli
birbirine yakın olmalı. Aradaki fark model, kurulum hatası mevzu burada!!

In [8]:
def log_ort(seri):
    return np.log1p(seri).mean() # RMSLE log uzayı ortalamayı da log da al


# 2025'in aylık seviyesi (gerçek) ve 2026 Mart (gerçek, train'in son ayı)
ay_2025 = train[train.yil == 2025].groupby('ay')['tuketim'].apply(log_ort)
mart_2026 = log_ort(train.loc[(train.yil == 2026) & (train.ay == 3), 'tuketim'])

# bizim 2026 tahminlerimizin aylık seviyesi
ay_tahmin = te.groupby('ay')['log_tahmin'].mean()

print('MART REFERANSLI KARŞILAŞTIRMA (log uzayında)')
print(f'{"ay":>4s} {"2025 gerçek":>12s} {"Marta göre":>11s} | '
      f'{"2026 tahmin":>12s} {"Marta göre":>11s}')
print(f'{3:4d} {ay_2025[3]:12.3f} {0.0:+11.3f} | {mart_2026:12.3f} {0.0:+11.3f}'
      f'   <- referans, ikisi de gerçek')
for a in [4, 5, 6, 7]:
    print(f'{a:4d} {ay_2025[a]:12.3f} {ay_2025[a]-ay_2025[3]:+11.3f} | '
          f'{ay_tahmin[a]:12.3f} {ay_tahmin[a]-mart_2026:+11.3f}')

print()
print(f'2025 Mart->Temmuz GERÇEK artış  : {ay_2025[7]-ay_2025[3]:+.3f} log')
print(f'2026 Mart->Temmuz BİZİM tahmin  : {ay_tahmin[7]-mart_2026:+.3f} log')
print(f'kaçırılan yaz artışı            : '
      f'{(ay_2025[7]-ay_2025[3])-(ay_tahmin[7]-mart_2026):+.3f} log '
      f'(yaklaşık %{(np.exp((ay_2025[7]-ay_2025[3])-(ay_tahmin[7]-mart_2026))-1)*100:.0f} '
      f'düşük tahmin)')

MART REFERANSLI KARŞILAŞTIRMA (log uzayında)
  ay  2025 gerçek  Marta göre |  2026 tahmin  Marta göre
   3        6.207      +0.000 |        6.431      +0.000   <- referans, ikisi de gerçek
   4        6.144      -0.063 |        6.440      +0.009
   5        6.100      -0.106 |        6.516      +0.085
   6        6.557      +0.351 |        6.786      +0.355
   7        7.030      +0.824 |        6.991      +0.560

2025 Mart->Temmuz GERÇEK artış  : +0.824 log
2026 Mart->Temmuz BİZİM tahmin  : +0.560 log
kaçırılan yaz artışı            : +0.264 log (yaklaşık %30 düşük tahmin)


2. Sebep: mevsimsellik trafodan trafoya çok değişiyor

Model ay özelliğini görüyor, yani ortalama bir mevsim eğrisi öğrenebiliyor.
Göremediği şey her trafonun kendi mevsim karakteri. Aşağıda her trafo için
yaz (Haz-Tem-Ağu) ile kış (Ara-Oca-Şub) log seviyeleri arasındaki farkı
hesaplayıp bu farkın trafolar arasında ne kadar dağıldığına bakıyoruz.

Eğer trafolar arası dağılım (std) global ortalamadan büyükse, modelin tek bir
ortalama eğri uygulaması ciddi bilgi kaybı demektir.

In [9]:
train['log_t'] = np.log1p(train['tuketim'])

yaz = train[train.ay.isin([6, 7, 8])].groupby('tanim')['log_t'].mean()
kis = train[train.ay.isin([12, 1, 2])].groupby('tanim')['log_t'].mean()
ortak = yaz.index.intersection(kis.index)
fark = yaz[ortak] - kis[ortak] # pozitif = yazın daha çok tüketiyor

print(f'{len(fark)} trafo için yaz - kış log farkı\n')
print(f'GLOBAL ortalama : {fark.mean():+.3f}  '
      f'<- modelin "ay" ile yakaladığı kısım')
print(f'trafolar arası std: {fark.std():.3f}  '
      f'<- modelin HİÇ göremediği kısım ({fark.std()/abs(fark.mean()):.1f} kat büyük)')
print()
print('dağılım:')
for q in [0.05, 0.25, 0.50, 0.75, 0.95]:
    v = fark.quantile(q)
    print(f'  %{q*100:>3.0f} dilim: {v:+.3f} log  -> yaz tüketimi kışın '
          f'{np.exp(v):.2f} katı')

# hangi bölgeler ayrışıyor
lokasyon = train.groupby('tanim')['lokasyon'].first()
bolge_fark = pd.DataFrame({'fark': fark, 'lokasyon': lokasyon[ortak]})
ozet = bolge_fark.groupby('lokasyon')['fark'].agg(['mean', 'size'])
ozet = ozet[ozet['size'] >= 40].sort_values('mean')
print()
print('en çok YAZ ağırlıklı 5 lokasyon (tarım/turizm):')
print(ozet.tail(5).round(3).to_string())
print('\nen çok KIŞ ağırlıklı 5 lokasyon (şehir merkezi):')
print(ozet.head(5).round(3).to_string())

2533 trafo için yaz - kış log farkı

GLOBAL ortalama : +0.227  <- modelin "ay" ile yakaladığı kısım
trafolar arası std: 0.977  <- modelin HİÇ göremediği kısım (4.3 kat büyük)

dağılım:
  %  5 dilim: -0.543 log  -> yaz tüketimi kışın 0.58 katı
  % 25 dilim: -0.066 log  -> yaz tüketimi kışın 0.94 katı
  % 50 dilim: +0.135 log  -> yaz tüketimi kışın 1.14 katı
  % 75 dilim: +0.370 log  -> yaz tüketimi kışın 1.45 katı
  % 95 dilim: +1.754 log  -> yaz tüketimi kışın 5.78 katı

en çok YAZ ağırlıklı 5 lokasyon (tarım/turizm):
                           mean  size
lokasyon                             
İZMİR>GÜNEY BÖLGE>ÇEŞME   0.402    74
İZMİR>GÜNEY BÖLGE>KİRAZ   0.462    96
MANİSA>ALAŞEHİR           0.594    58
İZMİR>GÜNEY BÖLGE>ÖDEMİŞ  0.977   103
MANİSA>GÖRDES             1.521    69

en çok KIŞ ağırlıklı 5 lokasyon (şehir merkezi):
                           mean  size
lokasyon                             
İZMİR>METROPOL>BORNOVA   -0.160   138
İZMİR>GÜNEY BÖLGE>URLA   -0.148    72
İZMİR>ME

 3. Düzeltme: aylık offset hesabı

Her ay için "olması gereken seviye"yi şöyle kuruyoruz:

Mart 2026'yı referans almanın önemli bir faydası var: yıllık büyümeyi otomatik
içeriyor. 2026'nın Ocak-Mart seviyesi 2025'ten +0.18 log yüksek ve bu kayma
referansa dahil oluyor, dolayısıyla ayrıca modellemeye gerek kalmıyor.

Varsayım: 2026 yazının şekli 2025 yazına benziyor. Tek yıllık veriye
dayandığı için bu bir risk. Aşağıda varsayımı elimizdeki tek imkanla, iki yılın
da tam olduğu Ocak-Mart dönemiyle kontrol ediyoruz: şekil birebir tekrar
etmiyor, aylık sapma ~0.08 log civarında. Ama düzeltmek istediğimiz Temmuz
sapması 0.264 log, yani bu belirsizliğin birkaç katı. Dolayısıyla düzeltmeyi
uygulamak yine de beklenen değer olarak kazançlı.

In [10]:
offset = {}
print(f'{"ay":>4s} {"beklenen":>10s} {"tahmin":>9s} {"offset":>9s} {"çarpan":>8s}')
for a in sorted(te.ay.unique()):
    beklenen = mart_2026 + (ay_2025[a] - ay_2025[3])
    offset[a] = beklenen - ay_tahmin[a]
    print(f'{a:4d} {beklenen:10.3f} {ay_tahmin[a]:9.3f} {offset[a]:+9.3f} '
          f'{np.exp(offset[a]):8.3f}')

print()
print('okuma: Temmuz tahminlerimiz olması gerekenin altında, Mayıs tahminlerimiz üstünde.')
print()

# varsayım kontrolü: 2026'nın Ocak-Mart şekli 2025'inkine ne kadar benziyor?
# elimizde iki yılın da tam olduğu tek dönem bu, dolayısıyla tek kontrol imkanı
print('VARSAYIM KONTROLÜ - iki yılın Ocak-Mart şekli (Şubat referanslı):')
sapmalar = []
for a in [1, 2, 3]:
    s25 = ay_2025[a] - ay_2025[2]
    s26 = (log_ort(train.loc[(train.yil == 2026) & (train.ay == a), 'tuketim'])
           - log_ort(train.loc[(train.yil == 2026) & (train.ay == 2), 'tuketim']))
    sapmalar.append(abs(s26 - s25))
    print(f'  ay {a}: 2025 {s25:+.3f} | 2026 {s26:+.3f} | fark {s26-s25:+.3f}')

belirsizlik = np.mean(sapmalar)
print(f'\n  şekil tekrarındaki ortalama sapma: {belirsizlik:.3f} log')
print(f'  Temmuz offsetimiz               : {offset[7]:.3f} log')
print(f'  -> şekil birebir tekrar etmiyor, ama offset bu belirsizliğin '
      f'{offset[7]/belirsizlik:.1f} katı.')
print(f'     Düzeltmeyi uygulamak yine de net kazanç: hata payı '
      f'{offset[7]**2:.3f} ten {belirsizlik**2:.3f} e iner.')

  ay   beklenen    tahmin    offset   çarpan
   4      6.368     6.440    -0.072    0.931
   5      6.325     6.516    -0.191    0.826
   6      6.782     6.786    -0.005    0.995
   7      7.255     6.991    +0.264    1.302

okuma: Temmuz tahminlerimiz olması gerekenin altında, Mayıs tahminlerimiz üstünde.

VARSAYIM KONTROLÜ - iki yılın Ocak-Mart şekli (Şubat referanslı):
  ay 1: 2025 -0.056 | 2026 +0.061 | fark +0.117
  ay 2: 2025 +0.000 | 2026 +0.000 | fark +0.000
  ay 3: 2025 -0.151 | 2026 -0.025 | fark +0.125

  şekil tekrarındaki ortalama sapma: 0.081 log
  Temmuz offsetimiz               : 0.264 log
  -> şekil birebir tekrar etmiyor, ama offset bu belirsizliğin 3.3 katı.
     Düzeltmeyi uygulamak yine de net kazanç: hata payı 0.070 ten 0.007 e iner.


4. Bu düzeltme skora ne kazandırır?

RMSLE karesi, hatanın sistematik (bias) ve rastgele (varyans)
bileşenlerine ayrılabilir. Bir aydaki ortalama sapma x ise, o ayın toplam
hataya katkısı x^2 kadar fazladır. Offset uygulamak tam olarak bu x^2
kısmını siler, varyansa dokunmaz.


In [11]:
KAGGLE_TABAN = 1.05487
mse = KAGGLE_TABAN ** 2
pay = te.groupby('ay').size() / len(te)

print(f'{"ay":>4s} {"satır payı":>11s} {"offset":>9s} {"MSE katkısı":>12s}')
kazanc = 0.0
for a in sorted(offset):
    katki = pay[a] * offset[a] ** 2
    kazanc += katki
    print(f'{a:4d} {pay[a]:11.3f} {offset[a]:+9.3f} {katki:12.4f}')

print()
print(f'mevcut MSE: {mse:.4f}   (RMSLE {KAGGLE_TABAN})')
print(f'sistematik sapmanın payı : {kazanc:.4f}   (toplam hatanın '
      f'%{kazanc/mse*100:.1f} i)')
print()
print('offset ağırlığına göre beklenen skor:')
for w in [0.0, 0.4, 0.6, 0.8, 1.0]:
    # w agirligiyla offset uygulaninca kalan bias (1-w)*offset olur
    kalan = sum(pay[a] * ((1 - w) * offset[a]) ** 2 for a in offset)
    print(f'  ağırlık {w:.1f} -> RMSLE {np.sqrt(mse - kazanc + kalan):.4f}')

  ay  satır payı    offset  MSE katkısı
   4       0.165    -0.072       0.0009
   5       0.256    -0.191       0.0093
   6       0.284    -0.005       0.0000
   7       0.295    +0.264       0.0206

mevcut MSE: 1.1128   (RMSLE 1.05487)
sistematik sapmanın payı : 0.0308   (toplam hatanın %2.8 i)

offset ağırlığına göre beklenen skor:
  ağırlık 0.0 -> RMSLE 1.0549
  ağırlık 0.4 -> RMSLE 1.0455
  ağırlık 0.6 -> RMSLE 1.0425
  ağırlık 0.8 -> RMSLE 1.0408
  ağırlık 1.0 -> RMSLE 1.0402


In [12]:
sample = pd.read_csv(RAW + 'sample_submission.csv')


def mevsim_duzelt(agirlik, dosya):
    """Tahminleri log uzayında ay bazlı offset ile kaydırıp kaydeder."""
    log_yeni = te['log_tahmin'] + agirlik * te['ay'].map(offset)
    yeni = pd.DataFrame({'id': te['id'],
                         'tuketim': np.clip(np.expm1(log_yeni), 0, None)})
    yeni = yeni.sort_values('id').reset_index(drop=True)

    saglam = (len(yeni) == len(sample)
              and set(yeni['id']) == set(sample['id'])
              and yeni['tuketim'].isna().sum() == 0
              and (yeni['tuketim'] < 0).sum() == 0)
    yeni.to_csv(f'../submissions/{dosya}.csv', index=False)

    sonrasi = np.log1p(yeni['tuketim']).groupby(
        te.sort_values('id')['ay'].values).mean()
    print(f'{dosya}  (offset x{agirlik})')
    print(f'  ortalama {yeni.tuketim.mean():7.1f} | medyan {yeni.tuketim.median():7.1f}'
          f' | doğrulama {"tamam" if saglam else "HATA"}')
    print('  düzeltme sonrası aylık log-ortalama: '
          + '  '.join(f'{a}. ay {v:.3f}' for a, v in sonrasi.items()))
    return yeni


sub15 = mevsim_duzelt(1.0, 'sub_15_mevsim_tam')
print()
sub16 = mevsim_duzelt(0.6, 'sub_16_mevsim_kismi')

print()
print('BEKLENEN aylık seviye (hedefimiz):     '
      + '  '.join(f'{a}. ay {mart_2026 + ay_2025[a] - ay_2025[3]:.3f}'
                  for a in sorted(offset)))
print()
print('İKİ DOSYA ARASINDAKİ FARK (ay bazında çarpan):')
print(f'{"ay":>4s} {"sub_15":>9s} {"sub_16":>9s} {"aradaki fark":>14s}')
for a in sorted(offset):
    c15, c16 = np.exp(offset[a]), np.exp(0.6 * offset[a])
    print(f'{a:4d} {c15:9.3f} {c16:9.3f} {(c15/c16-1)*100:+13.1f}%')
print('\nsub_16 aynı yönde ama daha az kaydırıyor; Temmuz farkı en belirgin.')

sub_15_mevsim_tam  (offset x1.0)
  ortalama  1975.7 | medyan  1249.9 | doğrulama tamam
  düzeltme sonrası aylık log-ortalama: 4. ay 6.368  5. ay 6.326  6. ay 6.782  7. ay 7.255

sub_16_mevsim_kismi  (offset x0.6)
  ortalama  1927.3 | medyan  1255.1 | doğrulama tamam
  düzeltme sonrası aylık log-ortalama: 4. ay 6.397  5. ay 6.401  6. ay 6.783  7. ay 7.149

BEKLENEN aylık seviye (hedefimiz):     4. ay 6.368  5. ay 6.325  6. ay 6.782  7. ay 7.255

İKİ DOSYA ARASINDAKİ FARK (ay bazında çarpan):
  ay    sub_15    sub_16   aradaki fark
   4     0.931     0.958          -2.8%
   5     0.826     0.892          -7.4%
   6     0.995     0.997          -0.2%
   7     1.302     1.172         +11.1%

sub_16 aynı yönde ama daha az kaydırıyor; Temmuz farkı en belirgin.
